# Trabajo Tema 4

Modelos de clasificación y clustering

Para la nota se tendrá en cuenta principalmente (80% de nota):
+ Comentarios de línea: La tarea básicamente es copiar código que ya existe en un libro o que os de una IA. Pero debéis entenderlo, explicando qué hace mediante comentarios.
+ Análisis de los resultados obtenidos en celdas Markdown.


# Parte 1: Clustering (6 puntos)

Esta parte trata de replicar y analizar los ejemplos resueltos del libro en PDF que está presente en el repositorio de GitHub

https://github.com/rpastorvargas/IntroductionToMachineLearningForSecurityPros

El código se proporciona (actualizado a Python 3, ya que el libro es antiguo) en el mismo repositorio, dentro del __directorio clustering_example__. 

El código está en archivos Python independientes, pero debéis entregar un Notebook donde se incluyan código, comentarios de línea y análisis/comparación de resultados en formato Markdown.

Como comentarios de análisis se deben indicar los resultados obtenidos en cada caso (K-Means y DBSCAN ) y compararlos con los correspondientes del libro. Es importante tener en cuenta que los resultados mostrados en el libro se corresponden con un subconjunto de los datos incluidos en el repositorio GitHub, por lo cual los resultados serán diferentes. 

El análisis y comparación se debe realizar de forma cualitativa para determinar si son coherentes y compatibles, fijandose más en los porcentajes en lugar de las cantidades absolutas.

También es importante repetir, al menos un par de veces, las ejecuciones de entrenamiento o ajuste (no las de vectorización o preparación de datos) para ver cómo afecta la aleatoriedad de los algoritmos y observar si hay algún cambio en los resultados.

## 1.1. CLUSTERING CON K-MEANS

El ejemplo comienza en página 23, "Cluster analysis with K-Means".

Debéis:
1. Copiar el código de cada archivo en su celda correspondiente. 
2. Analizar el código, entenderlo y __poner comentarios de línea__ explicando qué hace cada parte.
3. Ejecutar cada parte del código
4. Comentar el resultado obtenido y comparar los resultados con los del libro en celda Markdown.

Los archivos Python facilitados están construidos para ser ejecutados desde el terminal, pasando datos por parámetros. Debéis modificar y limpiar (lo que sobre) para que se ejecuten directamente desde el Notebook sin necesidad de recibir parámetros.

### Step 1: Vectorización y normalización

In [6]:
# Step 1: Vectorización y normalización de logs Apache
# ------------------------------------------------------
# Este código está adaptado al uso en notebook a partir del ejemplo del repositorio del libro.
# Objetivo: leer logs web, contar tipos de petición y códigos HTTP por IP,
# convertir cada IP en un vector numérico y normalizar los vectores.

import os                              # Permite recorrer carpetas y construir rutas de archivos
import re                              # Permite usar expresiones regulares para extraer campos del log
import socket                          # Convierte direcciones IP entre texto y formato binario
import struct                          # Convierte bytes de IP en enteros y viceversa
import h5py                            # Lee y escribe archivos HDF5 (.h5)
import numpy as np                     # Operaciones numéricas con arrays
from sklearn.preprocessing import normalize  # Normaliza vectores para que tengan magnitud comparable

# Ruta esperada de los logs. Si el notebook está en la carpeta clustering_example,
# esta ruta debe apuntar a data/www.secrepo.com/self.logs/
DATA_PATH = "data/www.secrepo.com/self.logs/"
OUTPUT_H5 = "secrepo.h5"

# Expresión regular para logs Apache.
# Extrae: IP, método HTTP y código de respuesta.
LOG_REGEX = re.compile(r'(\S+)\s\S+\s\S+\s\[[^\]]+\]\s"(\S*)\s[^"]*"\s(\d+)')


def ip2int(addr):
    """Convierte una IP en formato texto, por ejemplo '192.168.1.1', a entero."""
    return struct.unpack("!I", socket.inet_aton(addr))[0]


def get_prevectors(data_path=DATA_PATH, max_ips=10000):
    """Lee los logs y crea un diccionario con recuentos por IP.
    Cada IP tendrá dos grupos de características: métodos HTTP y códigos de respuesta.
    """
    # Se añaden estas IP desde el principio porque son las que aparecen en los ejemplos del libro.
    prevectors = {
        ip2int("192.187.126.162"): {"requests": {}, "responses": {}},
        ip2int("49.50.76.8"): {"requests": {}, "responses": {}},
        ip2int("70.32.104.50"): {"requests": {}, "responses": {}},
    }

    # Comprobación para evitar un error confuso si no se han descargado los datos.
    if not os.path.isdir(data_path):
        raise FileNotFoundError(
            f"No encuentro la carpeta de logs: {data_path}. "
            "Coloca los datos del repositorio en esa ruta o ajusta DATA_PATH."
        )

    # Recorremos todos los archivos de log de la carpeta.
    for filename in os.listdir(data_path):
        full_path = os.path.join(data_path, filename)
        if not os.path.isfile(full_path):
            continue

        # Se abren los logs ignorando caracteres raros para evitar errores de codificación.
        with open(full_path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                try:
                    # Extraemos IP, método HTTP y código de respuesta de cada línea.
                    ip, request_type, response_code = LOG_REGEX.findall(line)[0]
                    ip = ip2int(ip)
                    response_code = int(response_code)
                except IndexError:
                    # Si una línea no cumple el formato esperado, se ignora.
                    continue

                # Si aparece una IP nueva, se crea su entrada en el diccionario.
                if ip not in prevectors:
                    if len(prevectors) >= max_ips:
                        continue
                    prevectors[ip] = {"requests": {}, "responses": {}}

                # Sumamos una ocurrencia del método HTTP observado.
                prevectors[ip]["requests"][request_type] = prevectors[ip]["requests"].get(request_type, 0) + 1

                # Sumamos una ocurrencia del código de respuesta observado.
                prevectors[ip]["responses"][response_code] = prevectors[ip]["responses"].get(response_code, 0) + 1

    return prevectors


def convert_prevectors_to_vectors(prevectors):
    """Convierte el diccionario de recuentos en una matriz NumPy.
    Las columnas representan métodos HTTP y códigos de respuesta.
    """
    request_types = ["GET", "POST", "HEAD", "OPTIONS", "PUT", "TRACE"]
    response_codes = [200, 404, 403, 304, 301, 206, 418, 416, 400, 405, 503, 500]

    # Creamos una matriz de ceros: filas = IPs, columnas = características.
    vectors = np.zeros((len(prevectors), len(request_types) + len(response_codes)), dtype=np.float32)
    ips = []

    # Rellenamos cada fila con los recuentos de la IP correspondiente.
    for row_index, (ip, values) in enumerate(prevectors.items()):
        ips.append(ip)

        # Primer bloque de columnas: métodos HTTP.
        for col_index, request in enumerate(request_types):
            vectors[row_index, col_index] = values["requests"].get(request, 0)

        # Segundo bloque de columnas: códigos de respuesta.
        for col_index, code in enumerate(response_codes):
            vectors[row_index, len(request_types) + col_index] = values["responses"].get(code, 0)

    return np.array(ips), vectors


# Ejecución del Step 1.
try:
    prevectors = get_prevectors()
    ips, raw_vectors = convert_prevectors_to_vectors(prevectors)

    # Normalizamos para que las IPs con más peticiones no dominen únicamente por volumen.
    vectors = normalize(raw_vectors)

    # Guardamos vectores, etiquetas iniciales y notas/IPs en HDF5.
    with h5py.File(OUTPUT_H5, "w") as f:
        f.create_dataset("vectors", data=vectors)
        f.create_dataset("cluster", data=np.zeros((vectors.shape[0],), dtype=np.int32))
        f.create_dataset("notes", data=ips)

    print("Archivo creado:", OUTPUT_H5)
    print("Forma de la matriz de vectores:", vectors.shape)
except FileNotFoundError as e:
    print(e)
    print("Cuando tengas los logs en la ruta correcta, vuelve a ejecutar esta celda.")


No encuentro la carpeta de logs: data/www.secrepo.com/self.logs/. Coloca los datos del repositorio en esa ruta o ajusta DATA_PATH.
Cuando tengas los logs en la ruta correcta, vuelve a ejecutar esta celda.


#### Comentarios

En este primer paso se transforma información textual de logs web en datos numéricos que sí puede procesar un algoritmo de clustering. Cada fila representa una IP y cada columna representa una característica: número de peticiones `GET`, `POST`, etc., y número de respuestas `200`, `404`, `403`, etc.

La normalización es importante porque evita que una IP con muchísimas peticiones quede separada solo por tener más volumen. Lo que se busca comparar es el patrón de comportamiento, no únicamente la cantidad absoluta de solicitudes.

El resultado se guarda en `secrepo.h5`, un archivo HDF5 con tres datasets: `vectors`, `cluster` y `notes`. Inicialmente todos los clusters valen 0 porque todavía no se ha aplicado ningún algoritmo de agrupamiento.


### Step 2: Visualización de PCA

In [ ]:
# Step 2: Visualización de los vectores con PCA
# ----------------------------------------------
# PCA reduce la dimensionalidad de los vectores a 3 componentes para poder visualizarlos.

import h5py
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # Necesario para gráficos 3D en Matplotlib
from sklearn.decomposition import PCA


def visualize_vectors(h5_path="secrepo.h5"):
    """Carga los vectores desde HDF5, aplica PCA y dibuja una nube de puntos 3D."""
    with h5py.File(h5_path, "r") as f:
        vectors = f["vectors"][:]

    # PCA con 3 componentes permite representar los datos en un gráfico tridimensional.
    pca = PCA(n_components=3)
    projected_vectors = pca.fit_transform(vectors)

    print("Forma después de PCA:", projected_vectors.shape)
    print("Varianza explicada por cada componente:", pca.explained_variance_ratio_)
    print("Varianza explicada total:", pca.explained_variance_ratio_.sum())

    # Dibujamos los puntos en 3D.
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(projected_vectors[:, 0], projected_vectors[:, 1], projected_vectors[:, 2], s=5)
    ax.set_title("Visualización PCA de vectores de logs")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_zlabel("PC3")
    plt.show()

try:
    visualize_vectors("secrepo.h5")
except FileNotFoundError:
    print("No existe secrepo.h5. Ejecuta antes el Step 1 o coloca el archivo en la carpeta del notebook.")


#### Comentarios

PCA no hace clustering; solo reduce la dimensionalidad para que podamos observar visualmente si hay grupos o separaciones entre muestras. Si aparecen nubes de puntos separadas, puede ser una señal de que K-Means o DBSCAN encontrarán agrupaciones útiles.

La varianza explicada indica cuánta información aproximada conservan las tres componentes principales. Si el porcentaje total es bajo, la gráfica puede no representar bien toda la estructura real del espacio original.


### Step 3: First Pass Clustering with K-Means

In [ ]:
# Step 3: First Pass Clustering with K-Means
# ------------------------------------------
# Aplicamos K-Means con k=2, como primera prueba sencilla de agrupamiento.

import h5py
import numpy as np
from collections import Counter
from sklearn.cluster import KMeans


def run_kmeans(input_h5="secrepo.h5", output_h5="secrepo_kmeans_k2.h5", n_clusters=2, random_state=42):
    """Aplica K-Means y guarda un nuevo HDF5 con las etiquetas de cluster."""
    with h5py.File(input_h5, "r") as f:
        vectors = f["vectors"][:]
        ips = f["notes"][:]

    # n_init='auto' funciona en versiones nuevas de sklearn; si falla, se usa n_init=10.
    try:
        model = KMeans(n_clusters=n_clusters, random_state=random_state, n_init="auto")
        clusters = model.fit_predict(vectors)
    except TypeError:
        model = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
        clusters = model.fit_predict(vectors)

    # Contamos cuántas muestras quedan en cada cluster.
    counter = Counter(clusters.tolist())
    for label in sorted(counter):
        print(f"Label {label} has {counter[label]} samples")

    # Guardamos los mismos vectores, las etiquetas calculadas y las IPs.
    with h5py.File(output_h5, "w") as f:
        f.create_dataset("vectors", data=vectors)
        f.create_dataset("cluster", data=clusters.astype(np.int32))
        f.create_dataset("notes", data=ips)

    print("Archivo generado:", output_h5)
    return model, clusters

try:
    kmeans_k2_model, kmeans_k2_clusters = run_kmeans(n_clusters=2)
except FileNotFoundError:
    print("No existe secrepo.h5. Ejecuta antes el Step 1.")


#### Comentarios

K-Means intenta dividir los vectores en `k=2` grupos minimizando la distancia de cada muestra al centroide de su grupo. Es una primera aproximación útil, pero elegir `k=2` no garantiza que sea el número óptimo de clusters.

Si un cluster concentra casi todas las muestras y el otro tiene muy pocas, puede significar que hay un grupo mayoritario de comportamiento normal y un grupo pequeño con comportamiento diferente. También puede indicar que el valor de `k` no es suficiente para describir bien los datos.


#### Step 4: Validating Our Cluster Statistically

In [ ]:
# Step 4: Validating Our Cluster Statistically
# --------------------------------------------
# Calculamos estadísticas generales y métricas de validación, especialmente silhouette score.

import h5py
import socket
import struct
import numpy as np
from sklearn.metrics import pairwise_distances, pairwise_distances_chunked, silhouette_samples


def stats_pairs_distances(vectors):
    """Calcula distancia mínima, máxima y media entre vectores usando chunks para ahorrar memoria."""
    gen_dist_chunks = pairwise_distances_chunked(vectors)
    first_chunk = next(gen_dist_chunks)
    low = first_chunk.min()
    high = first_chunk.max()
    total = first_chunk.sum()
    count = first_chunk.size

    for chunk in gen_dist_chunks:
        low = min(low, chunk.min())
        high = max(high, chunk.max())
        total += chunk.sum()
        count += chunk.size

    # Restamos la diagonal de ceros, porque la distancia de cada vector consigo mismo no aporta información.
    return low, high, total / (count - np.sqrt(count))


def validate_clusters(h5_path="secrepo_kmeans_k2.h5"):
    """Muestra estadísticas del dataset y de cada cluster."""
    with h5py.File(h5_path, "r") as f:
        vectors = f["vectors"][:]
        clusters = f["cluster"][:]

    print("Vectors shape:", vectors.shape)
    print("Minimum feature value:", vectors.min())
    print("Mean feature value:", vectors.mean())
    print("Max feature value:", vectors.max())
    print("Percentage of null values:", 100.0 * float((vectors == 0).sum()) / vectors.size)
    print()

    low, high, avg = stats_pairs_distances(vectors)
    print("Minimum distance between vectors:", low)
    print("Mean distance between vectors:", avg)
    print("Maximum distance between vectors:", high)
    print()

    unique_labels = sorted(set(clusters.tolist()))
    print("Number of labels:", len(unique_labels))

    # Silhouette solo tiene sentido si hay al menos 2 clusters y menos clusters que muestras.
    if len(unique_labels) < 2 or len(unique_labels) >= len(vectors):
        print("No se puede calcular silhouette score con este número de etiquetas.")
        return

    silhouette_scores = silhouette_samples(vectors, clusters)
    centroid_distances = []

    for label in unique_labels:
        cluster_vectors = vectors[clusters == label, :]
        centroid = cluster_vectors.mean(axis=0)
        centroid_distances.extend(pairwise_distances(centroid.reshape(1, -1), cluster_vectors).tolist()[0])
        low, high, avg = stats_pairs_distances(cluster_vectors) if len(cluster_vectors) > 1 else (0, 0, 0)
        scores = silhouette_scores[clusters == label]
        print(
            f"Number of items in label {label}: {cluster_vectors.shape[0]} "
            f"({100.0 * cluster_vectors.shape[0] / vectors.shape[0]:.2f}%) "
            f"(avg dist: {avg}) (avg silhouette: {scores.mean()})"
        )

    centroid_distances = np.array(centroid_distances)
    print()
    print("Minimum label centroid distance:", centroid_distances.min())
    print("Mean label centroid distance:", centroid_distances.mean())
    print("Max label centroid distance:", centroid_distances.max())
    print("Overall Silhouette Score:", silhouette_scores.mean())

try:
    validate_clusters("secrepo_kmeans_k2.h5")
except FileNotFoundError:
    print("No existe secrepo_kmeans_k2.h5. Ejecuta antes el Step 3.")


#### Comentarios

La validación estadística ayuda a saber si los clusters están bien separados. La métrica `silhouette` compara qué tan cerca está una muestra de su propio cluster frente a otros clusters. Valores cercanos a 1 indican buena separación; valores cercanos a 0 indican solapamiento; valores negativos sugieren que algunas muestras podrían estar asignadas a un cluster incorrecto.

También se analizan distancias entre vectores y distancias al centroide. Si la distancia media dentro de un cluster es alta, ese cluster puede estar agrupando comportamientos bastante diferentes.


### Step 5: Inspecting Our Clusters

In [ ]:
# Step 5: Inspecting Our Clusters
# -------------------------------
# Mostramos las IPs asociadas a cada etiqueta de cluster.

import h5py
import socket
import struct


def int2ip(addr):
    """Convierte una IP almacenada como entero de vuelta a formato texto."""
    return socket.inet_ntoa(struct.pack("!I", int(addr)))


def inspect_clusters(h5_path="secrepo_kmeans_k2.h5", label=None, max_rows=50):
    """Imprime IPs por cluster. Si label es None, muestra todos los clusters."""
    with h5py.File(h5_path, "r") as f:
        ips = f["notes"][:]
        clusters = f["cluster"][:]

    labels_to_show = sorted(set(clusters.tolist())) if label is None else [label]

    shown = 0
    for cluster_id in labels_to_show:
        print(f"\nCluster {cluster_id}")
        for ip in ips[clusters == cluster_id]:
            print(cluster_id, int2ip(ip))
            shown += 1
            if shown >= max_rows:
                print(f"\nMostradas {max_rows} filas. Aumenta max_rows para ver más resultados.")
                return

try:
    inspect_clusters("secrepo_kmeans_k2.h5", label=None, max_rows=60)
except FileNotFoundError:
    print("No existe secrepo_kmeans_k2.h5. Ejecuta antes el Step 3.")


#### Comentarios

Inspeccionar las IPs permite interpretar el resultado del clustering. Las etiquetas `0`, `1`, etc. no tienen significado por sí mismas; lo importante es observar qué IPs se agrupan juntas y si tienen un comportamiento parecido.

Esta parte es útil para detectar posibles patrones anómalos. Por ejemplo, si un cluster pequeño contiene IPs con muchos errores `404` o métodos poco habituales, podría ser un grupo de actividad sospechosa.


### Step 6: Modifying K to Optimize Cluster Results

Volver a aplicar el algoritmo cambiando el valor de k. Copia de nuevo el código. 

In [ ]:
# Step 6: Modifying K to Optimize Cluster Results
# -----------------------------------------------
# Repetimos K-Means con un valor mayor de k para comprobar si mejora la separación.

# Probamos con k=3 como alternativa a k=2.
try:
    kmeans_k3_model, kmeans_k3_clusters = run_kmeans(
        input_h5="secrepo.h5",
        output_h5="secrepo_kmeans_k3.h5",
        n_clusters=3,
        random_state=42
    )
except FileNotFoundError:
    print("No existe secrepo.h5. Ejecuta antes el Step 1.")


#### Comentarios

Aumentar `k` permite crear más grupos y puede separar comportamientos que con `k=2` quedaban mezclados. Sin embargo, un `k` demasiado alto puede dividir artificialmente grupos que en realidad son similares.

La elección de `k` debe basarse en métricas como la inercia o silhouette, pero también en la interpretación del problema: se busca que los grupos resultantes tengan sentido desde el punto de vista del comportamiento de las IPs.


### Step 7: Repeating Our Inspection and Validation Procedures

In [ ]:
# Step 7: Repeating Our Inspection and Validation Procedures
# ----------------------------------------------------------
# Validamos e inspeccionamos los clusters generados con k=3.

try:
    print("=== Validación estadística para K-Means k=3 ===")
    validate_clusters("secrepo_kmeans_k3.h5")

    print("\n=== Inspección de IPs para K-Means k=3 ===")
    inspect_clusters("secrepo_kmeans_k3.h5", label=None, max_rows=60)
except FileNotFoundError:
    print("No existe secrepo_kmeans_k3.h5. Ejecuta antes el Step 6.")


#### Comentarios

Al repetir la validación se puede comparar `k=2` con `k=3`. Si el silhouette medio mejora y los clusters tienen tamaños razonables, `k=3` puede ser una mejor opción. Si el silhouette baja o aparecen clusters muy pequeños sin interpretación clara, probablemente el cambio no aporta mucho.

La inspección de IPs complementa las métricas. No basta con que una métrica mejore ligeramente: también se debe comprobar si los grupos resultantes son explicables.


### SELECCIÓN OPTIMA DE K MEDIANTE EL MÉTODO DEL CODO

Aplica K-Means en un bucle, desde k=2 a k=25, calculando la Inercia (WCSS) y graficandola (eje x es el valor de K, eje Y es la inercia)

In [ ]:
# Selección óptima de K mediante el método del codo
# -------------------------------------------------
# Probamos K-Means con k entre 2 y 25 y guardamos la inercia de cada modelo.

import h5py
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

try:
    with h5py.File("secrepo.h5", "r") as f:
        vectors = f["vectors"][:]

    k_values = range(2, 26)
    inertias = []

    for k in k_values:
        try:
            model = KMeans(n_clusters=k, random_state=42, n_init="auto")
        except TypeError:
            model = KMeans(n_clusters=k, random_state=42, n_init=10)
        model.fit(vectors)
        inertias.append(model.inertia_)
        print(f"k={k:2d} -> inercia={model.inertia_:.4f}")

    plt.figure(figsize=(8, 5))
    plt.plot(list(k_values), inertias, marker="o")
    plt.title("Método del codo para elegir k")
    plt.xlabel("Número de clusters, k")
    plt.ylabel("Inercia / WCSS")
    plt.xticks(list(k_values))
    plt.grid(True)
    plt.show()
except FileNotFoundError:
    print("No existe secrepo.h5. Ejecuta antes el Step 1.")


---
## 1.2. CLUSTERING CON DBSCAN

El ejemplo comienza en página 31, "Cluster analysis with DBSCAN".

Como bien dice el libro, podemos saltarnos step 1 y 2 ya que los datos están vectorizados y normalizados en secrepo.h5. Además no es necesario volver a graficar.

Pasamos directamente a aplicar el algoritmo.

Debéis:
1. Copiar el código de cada archivo en su celda correspondiente.
2. Analizar el código, entenderlo y __poner comentarios de línea__ explicando qué hace cada parte.
3. Ejecutar cada parte del código
4. Comentar el resultado obtenido y comparar los resultados con los del libro en celda Markdown.

### Ejecución para hiperparámetros Eps=0.5 y MinPts=5

Meter ejecución del algoritmo y la inspección de resultados en 2 celdas consecutivas

In [ ]:
# DBSCAN con Eps=0.5 y MinPts=5
# -----------------------------
# DBSCAN agrupa puntos densos y marca como ruido los puntos aislados con etiqueta -1.

import h5py
import numpy as np
from collections import Counter
from sklearn.cluster import DBSCAN


def run_dbscan(input_h5="secrepo.h5", output_h5="secrepo_dbscan_eps05_min5.h5", eps=0.5, min_samples=5):
    """Aplica DBSCAN y guarda las etiquetas de cluster en un nuevo HDF5."""
    with h5py.File(input_h5, "r") as f:
        vectors = f["vectors"][:]
        ips = f["notes"][:]

    model = DBSCAN(eps=eps, min_samples=min_samples)
    clusters = model.fit_predict(vectors)

    counter = Counter(clusters.tolist())
    for label in sorted(counter):
        print(f"Label {label} has {counter[label]} samples")

    with h5py.File(output_h5, "w") as f:
        f.create_dataset("vectors", data=vectors)
        f.create_dataset("cluster", data=clusters.astype(np.int32))
        f.create_dataset("notes", data=ips)

    print("Archivo generado:", output_h5)
    return model, clusters

try:
    dbscan_model, dbscan_clusters = run_dbscan(eps=0.5, min_samples=5)
except FileNotFoundError:
    print("No existe secrepo.h5. Ejecuta antes el Step 1.")


In [ ]:
# Inspección y validación de DBSCAN
# ---------------------------------
# Se reutilizan las funciones validate_clusters e inspect_clusters definidas en pasos anteriores.

try:
    print("=== Validación estadística para DBSCAN eps=0.5, min_samples=5 ===")
    validate_clusters("secrepo_dbscan_eps05_min5.h5")

    print("\n=== Inspección de IPs para DBSCAN ===")
    inspect_clusters("secrepo_dbscan_eps05_min5.h5", label=None, max_rows=60)
except FileNotFoundError:
    print("No existe secrepo_dbscan_eps05_min5.h5. Ejecuta antes la celda anterior.")


#### Comentarios

DBSCAN no necesita indicar el número de clusters, pero depende mucho de `eps` y `min_samples`. La etiqueta `-1` representa ruido, es decir, muestras que no pertenecen a ninguna zona suficientemente densa.

Con `eps=0.5` y `min_samples=5`, si aparecen demasiados puntos como ruido, `eps` puede ser demasiado pequeño o `min_samples` demasiado exigente. Si casi todo queda en un único cluster, `eps` puede ser demasiado grande.


### Pruebas de otros hiperparámetros

Aquí no seguimos el libro. Debéis pobrar varios hiperparámetros y decidir cual se ajusta mejor. Es decir, solo quedáis en la celda de código una versión. 

Por ejemplo:
1. Eps=0.6
2. Eps=0.4

Y variar también MinPts.

In [ ]:
# Pruebas de otros hiperparámetros para DBSCAN
# --------------------------------------------
# Probamos varias combinaciones y resumimos cuántos clusters y cuántos puntos de ruido produce cada una.

from collections import Counter
from sklearn.cluster import DBSCAN
import pandas as pd

try:
    with h5py.File("secrepo.h5", "r") as f:
        vectors = f["vectors"][:]
        ips = f["notes"][:]

    parameter_grid = [
        (0.3, 3),
        (0.4, 3),
        (0.4, 5),
        (0.5, 5),
        (0.6, 5),
        (0.7, 5),
        (0.8, 10),
    ]

    results = []
    best_clusters = None
    best_params = None

    for eps, min_samples in parameter_grid:
        model = DBSCAN(eps=eps, min_samples=min_samples)
        clusters = model.fit_predict(vectors)
        counts = Counter(clusters.tolist())

        # Número de clusters sin contar el ruido -1.
        n_clusters = len([label for label in counts if label != -1])
        noise_points = counts.get(-1, 0)
        noise_percent = 100.0 * noise_points / len(clusters)

        results.append({
            "eps": eps,
            "min_samples": min_samples,
            "n_clusters_sin_ruido": n_clusters,
            "puntos_ruido": noise_points,
            "% ruido": round(noise_percent, 2),
            "distribución": dict(sorted(counts.items()))
        })

        # Criterio sencillo: preferimos más de 1 cluster y no demasiado ruido.
        if n_clusters > 1 and noise_percent < 50 and best_clusters is None:
            best_clusters = clusters
            best_params = (eps, min_samples)

    results_df = pd.DataFrame(results)
    display(results_df)

    # Guardamos una versión elegida. Si ninguna cumple el criterio, usamos eps=0.6, min_samples=5.
    if best_clusters is None:
        chosen_eps, chosen_min_samples = 0.6, 5
        chosen_model = DBSCAN(eps=chosen_eps, min_samples=chosen_min_samples)
        best_clusters = chosen_model.fit_predict(vectors)
    else:
        chosen_eps, chosen_min_samples = best_params

    output_h5 = f"secrepo_dbscan_eps{str(chosen_eps).replace('.', '')}_min{chosen_min_samples}.h5"
    with h5py.File(output_h5, "w") as f:
        f.create_dataset("vectors", data=vectors)
        f.create_dataset("cluster", data=best_clusters.astype(np.int32))
        f.create_dataset("notes", data=ips)

    print("Parámetros elegidos:", {"eps": chosen_eps, "min_samples": chosen_min_samples})
    print("Archivo generado:", output_h5)
except FileNotFoundError:
    print("No existe secrepo.h5. Ejecuta antes el Step 1.")


#### Comentarios

El parámetro `eps` controla el radio de vecindad. Si `eps` aumenta, más puntos pasan a considerarse vecinos y los clusters tienden a crecer o fusionarse. Si `eps` disminuye, DBSCAN se vuelve más estricto y aparecen más puntos etiquetados como ruido (`-1`).

El parámetro `min_samples` indica cuántos puntos debe tener una zona para considerarse densa. Si se incrementa, se exige más densidad y pueden aparecer más puntos de ruido. Si se reduce, es más fácil formar clusters, aunque también se corre el riesgo de aceptar agrupaciones poco significativas.

La mejor configuración no es necesariamente la que produce más clusters, sino la que ofrece grupos interpretables y una cantidad razonable de ruido. En un contexto de seguridad, las muestras marcadas como ruido pueden ser especialmente interesantes porque representan comportamientos menos comunes.


---
# Parte 2: Regresión Logística (4 puntos)

En esta parte tendréis que investigar algo, no partimos de ningún libro.

El código entregado debe estar debidamente comentado, explicando qué es y qué hace cada instrucción, clase o método.

Puedes dividir en varias celdas según tu criterio.

Tareas a realizar:

1. Carga el dataset breast_cancer desde la librería sklearn.datasets.
2. Realiza una exploración inicial: ¿Cuántas muestras hay? ¿Cuántas características (features) definen cada caso? ¿están escaladas/normalizadas?
3. Divide los datos en dos conjuntos: 80% / 20%
4. Escala las características si no lo están. Debes aplicar también la transformación al conjunto de test.
5. Entrega el modelo LogisticRegression().
6. Genera y muestra la Matriz de Confusión.
7. Calcula métricas: Accuaracy_score, Precisión.
8. Analizar todos los resultados en los comentarios finales.

In [ ]:
# Parte 2: Regresión Logística con breast_cancer
# ----------------------------------------------
# Clasificamos tumores como malignos/benignos usando el dataset incluido en sklearn.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay

# 1. Cargamos el dataset breast_cancer.
data = load_breast_cancer()
X = data.data                       # Matriz de características: mediciones numéricas de cada muestra
y = data.target                     # Vector objetivo: 0 = malignant, 1 = benign
feature_names = data.feature_names  # Nombre de cada característica
class_names = data.target_names     # Nombre de las clases

# 2. Exploración inicial.
print("Número de muestras:", X.shape[0])
print("Número de características:", X.shape[1])
print("Clases:", dict(enumerate(class_names)))
print("Distribución de clases:")
print(pd.Series(y).map({0: class_names[0], 1: class_names[1]}).value_counts())

# Convertimos a DataFrame para facilitar la exploración.
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

display(df.head())
display(df.describe().T.head(10))

# 3. Dividimos los datos en entrenamiento y prueba.
# stratify=y mantiene la proporción de clases en ambos conjuntos.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 4. Estandarizamos las características.
# La regresión logística suele funcionar mejor si las variables tienen escalas comparables.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit solo con entrenamiento para evitar data leakage
X_test_scaled = scaler.transform(X_test)        # transform en test usando la misma media/desviación

# 5. Entrenamos el modelo.
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# 6. Realizamos predicciones.
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

# 7. Evaluación del modelo.
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
print("Accuracy:", round(accuracy, 4))
print("ROC AUC:", round(auc, 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=class_names))

# Matriz de confusión.
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(values_format="d")
plt.title("Matriz de confusión - Regresión logística")
plt.show()

# Curva ROC.
RocCurveDisplay.from_estimator(model, X_test_scaled, y_test)
plt.title("Curva ROC - Regresión logística")
plt.show()

# 8. Interpretación básica de coeficientes.
# Los coeficientes indican la influencia de cada variable en la predicción.
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coeficiente": model.coef_[0]
})
coef_df["abs_coeficiente"] = coef_df["coeficiente"].abs()
coef_df = coef_df.sort_values("abs_coeficiente", ascending=False)

display(coef_df.head(10))


#### Comentarios

El dataset `breast_cancer` contiene 569 muestras y 30 características numéricas calculadas a partir de imágenes de biopsias. La variable objetivo tiene dos clases: `malignant` y `benign`. No está perfectamente equilibrado, porque hay más casos benignos que malignos, pero la diferencia no impide entrenar un primer modelo.

Se divide el conjunto en entrenamiento y prueba para evaluar el modelo con datos no vistos. Además, se usa `stratify=y` para conservar la proporción de clases en ambos subconjuntos.

La estandarización es necesaria porque las características tienen escalas diferentes. Sin estandarizar, las variables con valores numéricos grandes podrían influir demasiado en el entrenamiento.

La regresión logística estima la probabilidad de pertenecer a una clase. En este caso se evalúa con `accuracy`, matriz de confusión, `classification_report` y `ROC AUC`. La matriz de confusión permite ver errores concretos: falsos positivos y falsos negativos. En un problema médico, los falsos negativos serían especialmente graves porque corresponderían a casos malignos clasificados como benignos.

Los coeficientes del modelo ayudan a interpretar qué variables influyen más en la decisión. Un coeficiente con valor absoluto alto indica una característica importante para separar las clases, aunque su interpretación exacta depende de cómo estén codificadas las clases.
